# Getting Started with Fracttalix Sentinel

This notebook walks through the core API: creating a detector, feeding observations,
inspecting alerts, and using the physics-derived diagnostics.

**Requirements:** `pip install fracttalix` (zero dependencies for core use).

In [ ]:
import math
import random

from fracttalix import SentinelDetector, SentinelConfig

## 1. Create a detector

Choose a preset config or build your own. `production()` is a good default.

In [ ]:
config = SentinelConfig.production()
det = SentinelDetector(config=config)

print(f"alpha={config.alpha}, multiplier={config.multiplier}, warmup={config.warmup_periods}")

## 2. Feed normal data

200 observations of a sine wave with small Gaussian noise. We expect very few false-positive alerts after warmup.

In [ ]:
random.seed(42)
FREQ = 2 * math.pi / 30

alerts = 0
for step in range(200):
    value = 2.0 * math.sin(FREQ * step) + random.gauss(0.0, 0.3)
    result = det.update_and_check(value)
    if result["alert"] and not result["warmup"]:
        alerts += 1

print(f"Normal phase complete: {alerts} false-positive alerts out of 200 observations")

## 3. Inject anomalies

Three large spikes that should be clearly detected.

In [ ]:
for spike_val in [25.0, -20.0, 30.0]:
    result = det.update_and_check(spike_val)
    if result["alert"]:
        print(f"ALERT at step {result['step']:>3d}  "
              f"value={spike_val:>7.1f}  "
              f"z_score={result['z_score']:.2f}  "
              f"reasons={result['alert_reasons']}")

## 4. Physics-derived diagnostics

Sentinel provides collapse-dynamics metrics beyond simple alerting.

In [ ]:
# Feed a few more normal values so the detector has recent context
for step in range(20):
    value = 2.0 * math.sin(FREQ * (220 + step)) + random.gauss(0.0, 0.3)
    result = det.update_and_check(value)

# Maintenance burden (Tainter regime)
mb = result.get_maintenance_burden()
print(f"Maintenance burden: mu={mb['mu']:.4f}, regime={mb['regime']}")

# Diagnostic window (time-to-collapse estimate)
dw = result.get_diagnostic_window()
print(f"Diagnostic window: steps={dw['steps']}, confidence={dw['confidence']}")

# Channel status
status = result.get_channel_status()
print(f"Channel status: {status}")

## 5. State persistence

Detector state can be serialized to JSON and restored.

In [ ]:
state_json = det.save_state()
print(f"State JSON size: {len(state_json):,} bytes")

# Restore into a new detector
det2 = SentinelDetector(config=config)
det2.load_state(state_json)
print(f"Restored detector — total observations: {len(det2)}")

## Next steps

- See `02_multistream.py` for thread-safe multi-stream monitoring
- See `03_collapse_detection.py` for PAC and intervention signatures
- See `04_autotune.py` for auto-tuning from labeled data
- Full API reference: [https://thomasbrennan.github.io/Fracttalix](https://thomasbrennan.github.io/Fracttalix)